# EMD-Based Male Depression Analysis

This notebook reproduces the analyses and figures from:
> *Analyzing Male Depression Using Empirical Mode Decomposition* (Sánchez Corrales et al., BIOSIGNALS 2025)

**Pipeline overview:**
1. Load pre-computed IMF arrays for the **male** cohort (depression vs. healthy)
2. Extract four per-segment statistical descriptors (skewness, kurtosis, std, mean) — no median
3. Bootstrap comparison with FDR correction → **Figure 2** (p-value heatmap)
4. Mean and standard deviation comparison across groups → **Figure 3** (probability density bar chart)
5. Gaussian Kernel (GK) discriminability criterion → **Figure 4** (GK similarity bar chart)

All paths and parameters are set in the **Configuration** section.

## 1. Imports

Load all required libraries for data handling, statistical analysis, and visualisation.
The `statsmodels` package is used for the Benjamini–Hochberg FDR correction.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import skew, kurtosis
from statsmodels.stats.multitest import multipletests

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

print("Libraries loaded successfully.")

## 2. Configuration

All user-defined parameters are centralised here. Adjust the paths to the `.npy` data files before running.

| Parameter | Description |
|---|---|
| `DEP_DATA_PATH` | Path to the depression-group IMF array |
| `HEALTH_DATA_PATH` | Path to the healthy-group IMF array |
| `N_IMFS` | Total number of IMFs per segment (default: 16) |
| `N_ITERATIONS` | Number of bootstrap iterations (paper: 10,000) |
| `ALPHA` | FDR significance threshold |
| `GK_SIGMA` | Bandwidth σ for the Gaussian kernel |
| `GK_THRESHOLD` | IMFs with mean GK similarity ≤ threshold are selected |
| `RANDOM_STATE` | Global random seed for reproducibility |

In [ ]:
# ── DATA PATHS (male cohort) ─────────────────────────────────────────────────
DEP_DATA_PATH    = "path/to/male_depression_imfs.npy"   # shape: (n_segments, N_IMFS, n_samples)
HEALTH_DATA_PATH = "path/to/male_healthy_imfs.npy"      # shape: (n_segments, N_IMFS, n_samples)

# ── GROUP / COHORT ───────────────────────────────────────────────────────────
GROUP_LABEL = "Male"   # used in plot titles

# ── IMF SETTINGS ─────────────────────────────────────────────────────────────
N_IMFS = 16

# ── BOOTSTRAP ────────────────────────────────────────────────────────────────
N_ITERATIONS = 10_000
ALPHA        = 0.05       # FDR significance level
RANDOM_STATE = 42

# ── GAUSSIAN KERNEL (GK) SELECTION ───────────────────────────────────────────
GK_SIGMA     = 0.5
GK_THRESHOLD = 1e-40

print(f"Configuration set — cohort: '{GROUP_LABEL}'")
print(f"  Depression data : {DEP_DATA_PATH}")
print(f"  Healthy data    : {HEALTH_DATA_PATH}")
print(f"  N_IMFS={N_IMFS} | Bootstrap iters={N_ITERATIONS} | α={ALPHA}")
print(f"  GK σ={GK_SIGMA} | GK threshold={GK_THRESHOLD}")

## 3. Data Loading

Load the pre-computed IMF arrays for the depression and healthy male groups from disk.

Expected array shape: **(n_segments, N_IMFS, n_samples)**
- `n_segments`: number of 20-second segments in each group (paper: 61 depression, 178 healthy)
- `N_IMFS`: number of EMD-derived Intrinsic Mode Functions (default: 16)
- `n_samples`: samples per segment (e.g. 200,000 at 10 kHz)

In [ ]:
dep_imfs    = np.load(DEP_DATA_PATH,    allow_pickle=True)
health_imfs = np.load(HEALTH_DATA_PATH, allow_pickle=True)

print(f"Depression group : {dep_imfs.shape}   (segments × IMFs × samples)")
print(f"Healthy group    : {health_imfs.shape}")

## 4. Convert to DataFrame

Reshape the 3-D arrays into 2-D DataFrames where each row is a segment and each column
contains the full signal array for one IMF. This format is required by the Gaussian kernel
comparison function (Section 7).

In [ ]:
imf_columns = [f"imf{i+1}" for i in range(N_IMFS)]

df_dep    = pd.DataFrame.from_records(dep_imfs,    columns=imf_columns)
df_health = pd.DataFrame.from_records(health_imfs, columns=imf_columns)

print(f"Depression DataFrame : {df_dep.shape}")
print(f"Healthy DataFrame    : {df_health.shape}")

## 5. Per-Segment Statistical Feature Extraction

For each segment and each IMF, four statistical descriptors are computed over the full
20-second signal:

| Descriptor | Function |
|---|---|
| Skewness | `scipy.stats.skew` |
| Kurtosis | `scipy.stats.kurtosis` (excess) |
| Standard deviation | `np.std` |
| Mean | `np.mean` |

Note: the median is **not** included (it was present in the paper's flowchart but is excluded
here following the statistical analysis described in the text, which uses the four descriptors above).

The result is one matrix per group with shape **(n_segments, N_IMFS)** for each statistic.

In [ ]:
STAT_NAMES = ['Skewness', 'Kurtosis', 'STD', 'Means']


def extract_segment_stats(imfs_3d):
    """
    Compute four per-segment statistics for each IMF.

    Parameters
    ----------
    imfs_3d : np.ndarray, shape (n_segments, N_IMFS, n_samples)

    Returns
    -------
    stats_dict : dict
        Keys are STAT_NAMES; each value is an ndarray of shape (n_segments, N_IMFS).
    """
    n_seg, n_imf, _ = imfs_3d.shape
    out = {name: np.zeros((n_seg, n_imf)) for name in STAT_NAMES}

    for s in range(n_seg):
        for i in range(n_imf):
            sig = imfs_3d[s, i, :]
            out['Skewness'][s, i] = skew(sig)
            out['Kurtosis'][s, i] = kurtosis(sig)
            out['STD'][s, i]      = np.std(sig)
            out['Means'][s, i]    = np.mean(sig)

    return out


print("Extracting statistics for depression group ...")
dep_stats = extract_segment_stats(dep_imfs)
print("Extracting statistics for healthy group ...")
health_stats = extract_segment_stats(health_imfs)

print(f"\nStat matrix shape (depression): {dep_stats['Means'].shape}   (segments × IMFs)")
print(f"Stat matrix shape (healthy)   : {health_stats['Means'].shape}")

## 6. Bootstrap Hypothesis Testing with FDR Correction

For each statistic and each IMF, a two-sided bootstrap test compares the group means:

1. **Observed difference**: `mean(dep) − mean(health)` for the statistic across segments.
2. **Bootstrap null distribution**: pool both groups, draw two resamples of the
   original sizes, compute their difference in means. Repeat `N_ITERATIONS` times.
3. **p-value**: fraction of bootstrap differences whose absolute value exceeds the
   observed absolute difference.
4. **FDR correction** (Benjamini–Hochberg): applied separately to the 16 p-values
   within each statistic to control the false discovery rate at level `ALPHA`.

This follows the procedure described in Section 2 of the paper.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)


def bootstrap_pvalue(dep_vals, health_vals, n_iter=N_ITERATIONS, rng=rng):
    """
    Two-sided bootstrap test comparing group means.

    Parameters
    ----------
    dep_vals, health_vals : 1-D arrays
        Statistic values (one per segment) for each group.
    n_iter : int
        Number of bootstrap iterations.

    Returns
    -------
    p_value : float
    """
    n_dep    = len(dep_vals)
    n_health = len(health_vals)
    pooled   = np.concatenate([dep_vals, health_vals])
    observed = np.abs(np.mean(dep_vals) - np.mean(health_vals))

    count = 0
    for _ in range(n_iter):
        rng.shuffle(pooled)
        boot_diff = np.abs(np.mean(pooled[:n_dep]) - np.mean(pooled[n_dep:]))
        if boot_diff >= observed:
            count += 1

    return count / n_iter


# ── Run bootstrap for all stats × all IMFs ───────────────────────────────────
raw_pvals   = {}   # stat → array of 16 raw p-values
corr_pvals  = {}   # stat → array of 16 FDR-corrected p-values
rejected    = {}   # stat → boolean array (True = significant after FDR)

for stat in STAT_NAMES:
    print(f"Bootstrap — {stat} ...", end=' ', flush=True)
    pvals = np.array([
        bootstrap_pvalue(dep_stats[stat][:, i], health_stats[stat][:, i])
        for i in range(N_IMFS)
    ])
    raw_pvals[stat] = pvals

    rej, p_corr, _, _ = multipletests(pvals, alpha=ALPHA, method='fdr_bh')
    corr_pvals[stat] = p_corr
    rejected[stat]   = rej

    sig_imfs = [imf_columns[i] for i, r in enumerate(rej) if r]
    print(f"significant IMFs: {sig_imfs if sig_imfs else 'none'}")

print("\nBootstrap + FDR complete.")

## 7. Figure 2 — P-value Heatmap (Bootstrap + FDR)

Replicates Figure 2 from the paper: a heatmap of FDR-corrected p-values for each
statistic (columns) × IMF (rows). Cells with p < α are highlighted in blue;
non-significant cells are white to light-grey. The numeric p-value is printed
inside each cell.

In [ ]:
# Build p-value matrix: rows = IMFs, columns = statistics
pval_matrix = np.column_stack([corr_pvals[s] for s in STAT_NAMES])   # (16, 4)
sig_matrix  = pval_matrix < ALPHA                                      # boolean mask

fig, ax = plt.subplots(figsize=(6, 8))

# Background colour: blue where significant, white elsewhere
bg = np.where(sig_matrix, pval_matrix, np.nan)
sns.heatmap(
    bg,
    ax=ax,
    cmap='Blues_r',
    vmin=0, vmax=ALPHA,
    linewidths=0.5, linecolor='#cccccc',
    xticklabels=STAT_NAMES,
    yticklabels=imf_columns,
    cbar_kws={'label': 'Corrected p-value (significant only)', 'shrink': 0.6},
    annot=False,
)

# Overlay all p-values as text
for row in range(N_IMFS):
    for col, stat in enumerate(STAT_NAMES):
        p = pval_matrix[row, col]
        color = 'white' if sig_matrix[row, col] and p < ALPHA / 2 else 'black'
        ax.text(
            col + 0.5, row + 0.5,
            f'{p:.3f}',
            ha='center', va='center',
            fontsize=8, color=color
        )

ax.set_title(
    f'Figure 2 — Bootstrap p-values with FDR correction ({GROUP_LABEL})\n'
    f'Blue cells: p < {ALPHA}',
    fontsize=12
)
ax.set_xlabel('Statistic', fontsize=11)
ax.set_ylabel('IMF', fontsize=11)
ax.tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.savefig('figure2_pvalue_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 2 saved → figure2_pvalue_heatmap.png")

## 8. Figure 3 — Mean and Std PDF Comparison

Replicates Figure 3 from the paper: a horizontal grouped bar chart comparing the average
mean and standard deviation of the IMF signals between the depression and healthy groups.

For each IMF, four bars are drawn:
- **Depression Mean** (dark red)
- **Healthy Mean** (salmon)
- **Depression Std** (dark blue)
- **Healthy Std** (light blue)

Values are plotted on a **log₁₀ scale** of absolute values (x-axis), consistent with
the paper's presentation. Bars extend horizontally from zero.

In [ ]:
# ── Group averages per IMF ────────────────────────────────────────────────────
dep_mean_avg    = np.mean(dep_stats['Means'], axis=0)    # (N_IMFS,)
health_mean_avg = np.mean(health_stats['Means'], axis=0)
dep_std_avg     = np.mean(dep_stats['STD'], axis=0)
health_std_avg  = np.mean(health_stats['STD'], axis=0)

# Log10 of absolute values (small negatives become large negative log values)
def safe_log10(arr):
    return -np.log10(np.abs(arr) + 1e-300)  # negative so bars go left

log_dep_mean    = safe_log10(dep_mean_avg)
log_health_mean = safe_log10(health_mean_avg)
log_dep_std     = safe_log10(dep_std_avg)
log_health_std  = safe_log10(health_std_avg)

# ── Plot ──────────────────────────────────────────────────────────────────────
y_pos  = np.arange(N_IMFS)       # one position per IMF
height = 0.18                     # bar thickness
offsets = [-1.5, -0.5, 0.5, 1.5]

colors = ['#8B0000', '#FA8072', '#00008B', '#87CEEB']  # dark red, salmon, dark blue, light blue
labels = ['Depression Mean', 'Healthy Mean', 'Depression Std', 'Healthy Std']
series = [log_dep_mean, log_health_mean, log_dep_std, log_health_std]

fig, ax = plt.subplots(figsize=(8, 10))

for offset, vals, color, label in zip(offsets, series, colors, labels):
    ax.barh(
        y_pos + offset * height,
        -vals,            # negate so bars extend to the LEFT
        height=height,
        color=color,
        label=label,
        edgecolor='none',
    )

ax.set_yticks(y_pos)
ax.set_yticklabels([f'IMF {i+1}' for i in range(N_IMFS)], fontsize=9)
ax.invert_yaxis()   # IMF 1 at the top
ax.set_xlabel('−log₁₀(|value|)', fontsize=11)
ax.set_title(
    f'Figure 3 — Mean and Std of IMF Probability Density Functions ({GROUP_LABEL})\n'
    'Comparison: Depression vs. Healthy',
    fontsize=12
)
ax.legend(loc='lower left', fontsize=9)
ax.axvline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig('figure3_mean_std_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 3 saved → figure3_mean_std_comparison.png")

## 9. Gaussian Kernel (GK) Discriminability Criterion

The **Radial Basis Function (RBF) Gaussian kernel** quantifies the cross-class distributional
similarity for each IMF:

$$K(x, x') = \exp\!\left(-\frac{\|x - x'\|^2}{2\sigma^2}\right)$$

For each IMF, all pairwise combinations of one depression segment and one healthy segment
are compared. The **mean kernel value** measures overall cross-class similarity:
a lower value indicates greater distributional discrepancy (more discriminative IMF).

IMFs with mean similarity ≤ `GK_THRESHOLD` are marked as selected.

In [ ]:
def gaussian_kernel(x, y, sigma):
    """Point-wise RBF Gaussian kernel between two 1-D arrays (truncated to equal length)."""
    min_len = min(len(x), len(y))
    x, y = x[:min_len], y[:min_len]
    return np.exp(-np.sum((x - y) ** 2) / (2 * sigma ** 2))


def compute_gk_similarities(df_dep, df_health, sigma=GK_SIGMA):
    """
    Compute the mean cross-class Gaussian kernel similarity for each IMF.

    Parameters
    ----------
    df_dep, df_health : pd.DataFrame
        Segment-level DataFrames (one column per IMF; each cell is a 1-D signal array).
    sigma : float
        Bandwidth parameter for the RBF kernel.

    Returns
    -------
    similarities : list of float  (length = N_IMFS)
    """
    n_imfs = df_dep.shape[1]
    similarities = []

    for i in range(n_imfs):
        sims = [
            gaussian_kernel(df_dep.iloc[r, i], df_health.iloc[s, i], sigma)
            for r in range(len(df_dep))
            for s in range(len(df_health))
        ]
        similarities.append(np.mean(sims))

    return similarities


print(f"Computing GK similarities (σ={GK_SIGMA}) — this may take a few minutes ...")
gk_similarities = compute_gk_similarities(df_dep, df_health)

selected_imf_idx   = [i for i, s in enumerate(gk_similarities) if s <= GK_THRESHOLD]
selected_imf_names = [imf_columns[i] for i in selected_imf_idx]

print("\nGK similarities per IMF:")
for name, sim in zip(imf_columns, gk_similarities):
    marker = " ← SELECTED" if sim <= GK_THRESHOLD else ""
    print(f"  {name}: {sim:.4e}{marker}")

print(f"\nSelected IMFs : {selected_imf_names}")

## 10. Figure 4 — Gaussian Kernel Similarity Bar Chart

Replicates Figure 4 from the paper: a horizontal bar chart of the mean cross-class
Gaussian kernel similarity for each IMF, plotted on a **logarithmic x-axis**.

IMFs with lower similarity (further left) are more discriminative between depression
and healthy groups. The paper identifies IMFs 2–6 as the most discriminative in men.

In [ ]:
y_pos = np.arange(N_IMFS)

fig, ax = plt.subplots(figsize=(7, 8))

ax.barh(
    y_pos,
    gk_similarities,
    color='#FA8072',      # salmon, as in the paper
    edgecolor='none',
    height=0.6,
)

ax.set_yticks(y_pos)
ax.set_yticklabels([f'IMF {i+1}' for i in range(N_IMFS)], fontsize=9)
ax.invert_yaxis()         # IMF 1 at the top

ax.set_xscale('log')
ax.set_xlabel('Mean Gaussian Kernel Similarity (log scale)', fontsize=11)
ax.set_title(
    f'Figure 4 — Gaussian Kernel Cross-Class Similarity per IMF ({GROUP_LABEL})\n'
    f'Lower = more discriminative | GK threshold = {GK_THRESHOLD:.0e}',
    fontsize=12
)

# Mark selected IMFs
for i in selected_imf_idx:
    ax.get_yticklabels()[i].set_color('#8B0000')
    ax.get_yticklabels()[i].set_fontweight('bold')

if GK_THRESHOLD > 0:
    ax.axvline(GK_THRESHOLD, color='#333333', linestyle='--',
               linewidth=1.2, label=f'Threshold ({GK_THRESHOLD:.0e})')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('figure4_gk_similarity.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 4 saved → figure4_gk_similarity.png")

## 11. Summary

| Output | File | Description |
|---|---|---|
| Figure 2 | `figure2_pvalue_heatmap.png` | FDR-corrected bootstrap p-values per IMF × statistic |
| Figure 3 | `figure3_mean_std_comparison.png` | Mean and std PDF comparison: depression vs. healthy |
| Figure 4 | `figure4_gk_similarity.png` | Gaussian Kernel cross-class similarity per IMF |

**Selected IMFs** (GK criterion): printed in bold red on Figure 4's y-axis and listed below.

In [ ]:
print(f"=== Analysis complete — {GROUP_LABEL} cohort ===")
print(f"\nBootstrap + FDR significant IMFs (α={ALPHA}):")
for stat in STAT_NAMES:
    sig = [imf_columns[i] for i, r in enumerate(rejected[stat]) if r]
    print(f"  {stat:12s}: {sig if sig else 'none'}")

print(f"\nGK-selected IMFs (threshold={GK_THRESHOLD:.0e}): {selected_imf_names}")

df_summary = pd.DataFrame({
    'IMF': imf_columns,
    'GK_similarity': [f'{s:.4e}' for s in gk_similarities],
    'GK_selected': ['Yes' if i in selected_imf_idx else 'No' for i in range(N_IMFS)],
    **{f'p_{stat}': [f'{corr_pvals[stat][i]:.3f}' for i in range(N_IMFS)] for stat in STAT_NAMES},
    **{f'sig_{stat}': ['*' if rejected[stat][i] else '' for i in range(N_IMFS)] for stat in STAT_NAMES},
})

print("\nFull results table:")
display(df_summary)